# Démonstration d'inférence U-TILISE

Ce notebook montre comment charger un modèle entraîné et produire des reconstructions
sans nuages sur un échantillon du jeu de test.

**Contenu :**
1. Chargement du modèle depuis un checkpoint
2. Préparation d'un échantillon de test
3. Inférence et visualisation des résultats
4. Visualisation des masques d'attention
5. Calcul des métriques de reconstruction

> **Prérequis :** l'environnement conda `cloud_reconstruction` doit être activé.
> Un checkpoint entraîné et le fichier HDF5 doivent être accessibles.

In [ ]:
import os
import sys

# Se placer à la racine du projet
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Répertoire de travail : {PROJECT_ROOT}")
# Charger les variables d'environnement depuis .env
from dotenv import load_dotenv
load_dotenv()


## 1. Configuration

Définir les chemins vers le checkpoint et la config d'entraînement.

In [ ]:
# Les chemins sont lus depuis le fichier .env (voir .env.example)
CHECKPOINT = os.environ["CHECKPOINT"]
CONFIG_TRAIN = os.environ["TRAIN_CONFIG"]
HDF5_FILE = os.environ["HDF5_FILE"]

# Paramètres d'inférence
TEMPORAL_WINDOW = 14   # Taille de la fenêtre glissante
BLEND_MODE = "center"  # Mode de fusion : switch / center / center_only / iterative

print(f"Checkpoint      : {CHECKPOINT}")
print(f"Config train    : {CONFIG_TRAIN}")
print(f"HDF5            : {HDF5_FILE}")
print(f"Fenêtre         : {TEMPORAL_WINDOW}")
print(f"Mode de fusion  : {BLEND_MODE}")


## 2. Chargement du modèle

La classe `Imputation` gère le chargement du checkpoint et l'inférence par fenêtre glissante.

In [ ]:
import torch
from src.eval_tools import Imputation

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}\n")

imputer = Imputation(
    config_file_train=CONFIG_TRAIN,
    method="utilise",
    checkpoint=CHECKPOINT,
    temporal_window=TEMPORAL_WINDOW,
    blend_mode=BLEND_MODE,
    device=device,
)

## 3. Préparation d'un échantillon de test

In [ ]:
from omegaconf import OmegaConf
from src import config_utils, data_utils

# Charger la config et préparer le dataset de test
cfg_default = config_utils.read_config("configs/default.yaml")
cfg_eval = config_utils.read_config("configs/config_run_eval.yaml")
config = OmegaConf.merge(cfg_default, cfg_eval)

# Mettre à jour le chemin HDF5
config.data.hdf5_file = HDF5_FILE

test_dset = data_utils.get_dataset(config, phase="test")
print(f"Nombre d'échantillons de test : {len(test_dset)}")

# Charger un échantillon
SAMPLE_IDX = 0  # Modifier cet indice pour explorer d'autres patches
test_loader = data_utils.get_dataloader(
    test_dset, config, drop_last=False, shuffle=False, subset=5,
)
batch = next(iter(test_loader))

print(f"\nÉchantillon chargé :")
print(f"  Entrée (x)   : {batch['x'].shape}  — (B, T, C, H, W)")
print(f"  Cible (y)    : {batch['y'].shape}")
print(f"  Masques      : {batch['masks'].shape}")

## 4. Inférence et visualisation

L'inférence utilise une fenêtre glissante avec le mode de fusion configuré.
On récupère aussi les cartes d'attention pour les visualiser ensuite.

In [ ]:
%matplotlib inline
from src import visutils
from src.data_utils import extract_sample

# Inférence avec retour des cartes d'attention
batch_out, y_pred, att = imputer.impute_sample(batch, return_att=True)

inputs, target, masks, mask_valid, cloud_mask, indices_rgb, index_nir = extract_sample(batch_out)
idx_rgb = indices_rgb.int().tolist()

print(f"Prédiction : {y_pred.shape}")
print(f"Attention  : {att.shape}  — (n_head, B, T, T, h, w)")

In [ ]:
import matplotlib.pyplot as plt

# --- Galerie : entrée masquée ---
fig_input = visutils.sequence2gallery(
    inputs[0], variable="rgb", indices_rgb=idx_rgb, brightness_factor=3.0,
)
fig_input.suptitle("Entrée masquée (input)", y=1.02)
plt.show()

# --- Galerie : cible (ground truth) ---
fig_target = visutils.sequence2gallery(
    target[0], variable="rgb", indices_rgb=idx_rgb, brightness_factor=3.0,
)
fig_target.suptitle("Cible (ground truth)", y=1.02)
plt.show()

# --- Galerie : prédiction ---
fig_pred = visutils.sequence2gallery(
    y_pred[0], variable="rgb", indices_rgb=idx_rgb, brightness_factor=3.0,
)
fig_pred.suptitle("Prédiction U-TILISE", y=1.02)
plt.show()

In [ ]:
# --- Galerie : masques de nuages ---
fig_masks = visutils.sequence2gallery(
    masks[0], variable="binary_mask", brightness_factor=1.0,
)
fig_masks.suptitle("Masques appliqués (1 = masqué)", y=1.02)
plt.show()

## 5. Visualisation des masques d'attention

Le modèle U-TILISE utilise un Lightweight Temporal Attention Encoder (LTAE) avec
plusieurs têtes d'attention. Chaque tête apprend à pondérer différemment les dates
d'entrée pour reconstruire chaque date cible.

In [ ]:
from src.eval_tools import visualize_att_for_target_t_across_heads

# Visualiser les masques de toutes les têtes d'attention pour une date cible
T_TARGET = 3  # Indice de la date cible à visualiser

fig_att = visualize_att_for_target_t_across_heads(
    seq=inputs, att=att,
    t_target=T_TARGET,
    batch=0,
    indices_rgb=idx_rgb,
    brightness_factor=3.0,
)
plt.show()

In [ ]:
from src.eval_tools import visualize_att_for_one_head_across_time

# Visualiser une tête d'attention donnée pour toutes les dates
HEAD = 0  # Indice de la tête d'attention

fig_att_head = visualize_att_for_one_head_across_time(
    seq=inputs, att=att,
    head=HEAD,
    batch=0,
    indices_rgb=idx_rgb,
    brightness_factor=3.0,
)
plt.show()

## 6. Calcul des métriques

Métriques de reconstruction calculées sur les pixels masqués et observés :
MAE, RMSE, PSNR, SSIM, SAM.

In [ ]:
from src.metrics.cloud_removal import CloudRemovalMetrics

compute_metrics = CloudRemovalMetrics(
    metrics=["mae", "rmse", "psnr", "ssim", "sam"],
    eval_occluded_observed=True,
)

# Calcul des métriques pour le premier échantillon du batch
metrics = compute_metrics(
    y_pred[0:1],
    target[0:1],
    masks[0:1],
    mask_valid[0:1] if mask_valid is not None else None,
)

print("=== Métriques de reconstruction ===")
for name, value in metrics.items():
    if isinstance(value, (float, int)):
        print(f"  {name:30s} : {value:.4f}")
    elif hasattr(value, 'item'):
        print(f"  {name:30s} : {value.item():.4f}")

---

**Pour une évaluation complète** sur tout le jeu de test :

```bash
python run_eval.py configs/config_run_eval.yaml utilise
```

**Pour l'inférence à la tuile** (production de GeoTIFF) :

```bash
python run_inference.py configs/config_run_inference.yaml utilise
```

Voir le README pour les détails sur les modes de masquage et de fusion temporelle.